# ADNI Project: AD vs HC Classification + Unsupervised Learning on HC

Group notebook — combines Task A (unsupervised learning on HC) and Task B (AD vs HC classification), sharing one masking + AAL feature-extraction pipeline.

Imports the reusable functions from `src/` instead of duplicating code per group member.

## 0. Setup

In [ ]:
import sys
sys.path.append("src")

import numpy as np
import pandas as pd
import nibabel as nib
import matplotlib.pyplot as plt

import config
from data_loading import list_nifti_files, extract_subject_id, check_image_consistency
from preprocessing import load_volumes, apply_mask, zscore_normalize
from feature_extraction import resample_atlas_to_reference, extract_roi_features
from task_a_unsupervised import run_pca, cluster_kmeans, cluster_hierarchical, stability_check
from task_b_classification import build_combined_dataset, run_cross_validation, get_classifiers
from evaluation import plot_roc_curves, plot_silhouette_scores

config.OUTPUT_DIR.mkdir(exist_ok=True)
config.FIGURES_DIR.mkdir(parents=True, exist_ok=True)
config.RESULTS_DIR.mkdir(parents=True, exist_ok=True)

np.random.seed(config.RANDOM_SEED)


## 1. Load file lists and check consistency

In [ ]:
hc_mri_files = list_nifti_files(config.HC_MRI_DIR)
ad_mri_files = list_nifti_files(config.AD_MRI_DIR)

print(f"HC MRI files: {len(hc_mri_files)}")
print(f"AD MRI files: {len(ad_mri_files)}")

metadata_df, consistency_summary = check_image_consistency({
    "HC MRI": hc_mri_files,
    "AD MRI": ad_mri_files,
})
display(consistency_summary)


In [ ]:
# Confirm no subject overlap between AD and HC groups
hc_ids_check = set(extract_subject_id(f) for f in hc_mri_files)
ad_ids_check = set(extract_subject_id(f) for f in ad_mri_files)
overlap = hc_ids_check & ad_ids_check

print(f"Subject overlap between AD and HC: {len(overlap)}")
assert len(overlap) == 0, "Overlapping subjects found between AD and HC — investigate before continuing!"


## 2. Load ICV mask and resample AAL atlas to MRI space

In [ ]:
rmask_img = nib.load(config.RMASK_PATH)
rmask_data = rmask_img.get_fdata()

reference_img = nib.load(hc_mri_files[0])
atlas_data = resample_atlas_to_reference(config.AAL_PATH, reference_img)

roi_labels = [int(l) for l in np.unique(atlas_data) if l != 0]
print(f"Number of ROI labels in AAL atlas: {len(roi_labels)}")


## 3. Preprocess HC: mask + normalize

In [ ]:
hc_volumes = load_volumes(hc_mri_files)
hc_masked = apply_mask(hc_volumes, rmask_data)
hc_normalized, hc_zero_std = zscore_normalize(hc_masked)

print(f"HC normalized volumes shape: {hc_normalized.shape}")
if hc_zero_std:
    print(f"WARNING - HC subjects with zero-std intensity (check these scans): {hc_zero_std}")


## 4. Preprocess AD: mask + normalize

Same pipeline as HC, applied to the AD group (this was the missing piece from the original notebook).

In [ ]:
ad_volumes = load_volumes(ad_mri_files)
ad_masked = apply_mask(ad_volumes, rmask_data)
ad_normalized, ad_zero_std = zscore_normalize(ad_masked)

print(f"AD normalized volumes shape: {ad_normalized.shape}")
if ad_zero_std:
    print(f"WARNING - AD subjects with zero-std intensity (check these scans): {ad_zero_std}")


## 5. Extract AAL ROI features for both groups

In [ ]:
hc_features, hc_missing = extract_roi_features(hc_normalized, atlas_data, roi_labels)
ad_features, ad_missing = extract_roi_features(ad_normalized, atlas_data, roi_labels)

hc_subject_ids = [extract_subject_id(f) for f in hc_mri_files]
ad_subject_ids = [extract_subject_id(f) for f in ad_mri_files]

print(f"HC ROI feature matrix: {hc_features.shape}, missing observations: {len(hc_missing)}")
print(f"AD ROI feature matrix: {ad_features.shape}, missing observations: {len(ad_missing)}")


---
## TASK A: Unsupervised Learning on HC

In [ ]:
reduced_hc, pca_model, hc_scaler = run_pca(hc_features)
print(f"Reduced HC feature shape after PCA: {reduced_hc.shape}")
print(f"Variance explained: {pca_model.explained_variance_ratio_.sum():.3f}")


In [ ]:
silhouette_df = cluster_kmeans(reduced_hc)
display(silhouette_df)
plot_silhouette_scores(silhouette_df, save_path=config.FIGURES_DIR / "silhouette_scores.png")

best_k = int(silhouette_df.loc[silhouette_df["silhouette"].idxmax(), "k"])
print(f"Selected k = {best_k} (highest silhouette score)")


In [ ]:
hc_cluster_labels = cluster_hierarchical(reduced_hc, n_clusters=best_k)

hc_cluster_df = pd.DataFrame({
    "Subject": hc_subject_ids,
    "Cluster": hc_cluster_labels,
})
display(hc_cluster_df["Cluster"].value_counts())


In [ ]:
stability = stability_check(reduced_hc, n_clusters=best_k, n_seeds=10)
print(f"Cluster stability - mean ARI: {stability['mean_ari']:.3f} (+/- {stability['std_ari']:.3f})")
print("An ARI close to 1.0 means the clustering is stable across different random seeds.")


---
## TASK B: AD vs HC Classification

In [ ]:
combined_df = build_combined_dataset(
    hc_features, ad_features, hc_subject_ids, ad_subject_ids, roi_labels
)
combined_df.to_csv(config.RESULTS_DIR / "combined_roi_features.csv", index=False)

print(f"Combined dataset shape: {combined_df.shape}")
display(combined_df["Label"].value_counts().rename({0: "HC", 1: "AD"}))
display(combined_df.head())


In [ ]:
X = combined_df.drop(columns=["Subject", "Label"]).values
y = combined_df["Label"].values

# Simple mean-imputation for any missing ROI values before classification
col_means = np.nanmean(X, axis=0)
inds = np.where(np.isnan(X))
X[inds] = np.take(col_means, inds[1])

print(f"Missing values remaining: {np.isnan(X).sum()}")


In [ ]:
results_df, summary, roc_curves = run_cross_validation(X, y, n_splits=config.N_CV_FOLDS, random_state=config.RANDOM_SEED)
display(summary)

results_df.to_csv(config.RESULTS_DIR / "classification_cv_results.csv", index=False)
plot_roc_curves(roc_curves, save_path=config.FIGURES_DIR / "roc_curves.png")


In [ ]:
# Per-fold breakdown, useful for the robustness/stability part of your report
display(results_df.pivot_table(index="fold", columns="model", values="auc"))


---
## Linking Task A and Task B

Check whether HC subjects that are hardest to classify correctly in Task B (i.e. that a classifier tends to confuse with AD) correspond to the small outlier cluster found in Task A. This is the "related tasks" justification for the report.

In [ ]:
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

# Refit a single logistic regression with cross-validated predictions per subject
skf = StratifiedKFold(n_splits=config.N_CV_FOLDS, shuffle=True, random_state=config.RANDOM_SEED)
oof_pred = np.zeros(len(y))

for train_idx, test_idx in skf.split(X, y):
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X[train_idx])
    X_test = scaler.transform(X[test_idx])

    clf = LogisticRegression(class_weight="balanced", max_iter=2000, random_state=config.RANDOM_SEED)
    clf.fit(X_train, y[train_idx])
    oof_pred[test_idx] = clf.predict(X_test)

combined_df["Predicted"] = oof_pred.astype(int)

# Subset to HC subjects that were misclassified as AD
hc_mask = combined_df["Label"] == 0
misclassified_hc = combined_df.loc[hc_mask & (combined_df["Predicted"] == 1), "Subject"]

print(f"HC subjects misclassified as AD: {len(misclassified_hc)}")
print(misclassified_hc.tolist())


In [ ]:
# Cross-reference against Task A's cluster assignments
misclassified_clusters = hc_cluster_df[hc_cluster_df["Subject"].isin(misclassified_hc)]
display(misclassified_clusters)

print("\nCluster distribution of misclassified HC subjects vs. overall HC cluster distribution:")
print(misclassified_clusters["Cluster"].value_counts(normalize=True))
print(hc_cluster_df["Cluster"].value_counts(normalize=True))


If misclassified HC subjects are over-represented in one cluster (especially a small outlier cluster), that supports the narrative that Task A's clustering identifies atypical/borderline HC subjects that are also harder for Task B's classifier to distinguish from AD — this is the explicit link between the two tasks for your report.

## 6. Summary for the report

- **Preprocessing**: ICV masking + subject-wise z-score normalization, applied identically to AD and HC.
- **Feature extraction**: AAL atlas (nearest-neighbor resampled to MRI space), mean intensity per ROI (116 regions).
- **Task A**: PCA + KMeans/hierarchical clustering on HC, with `k` chosen via silhouette score and validated for stability via multi-seed ARI.
- **Task B**: Logistic Regression / SVM / MLP, 5-fold stratified CV, scaling fit inside each fold to avoid leakage and preserve the AD-vs-HC signal. Reported via AUC, sensitivity, specificity rather than raw accuracy (imbalanced classes).
- **Link between tasks**: cross-referenced Task B's misclassified HC subjects against Task A's cluster assignments.